## load data in Document objects with lang chain 

In [23]:
from langchain_community.document_loaders import TextLoader

In [81]:

loader = TextLoader("data/txt/python.txt", encoding="utf-8")
documents = loader.load()

print(documents)

[Document(metadata={'source': 'data/txt/python.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python emphasizes clean syntax and ease of use, making it an excellent choice for beginners and experienced developers alike. Its design philosophy encourages writing clear and concise code, which reduces the complexity often associated with programming.\n\nPython is widely used in various fields such as web development, data science, artificial intelligence, machine learning, automation, and scientific computing. Popular frameworks like Django and Flask support web development, while libraries such as NumPy, pandas, and TensorFlow make Python a powerful tool for data analysis and machine learning.\n\nOne of Python’s key strengths is its large and active community, which contributes to a vast ecosystem of libraries and tools. This allows developers to quickly build a

## directory loader

In [82]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "data/txt",
    glob="**/*.txt", ##pattern to match files
    loader_cls=TextLoader, ##loader class to use
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True
)

directory_documnets = dir_loader.load()
directory_documnets

100%|██████████| 2/2 [00:00<00:00, 90.52it/s]


[Document(metadata={'source': 'data\\txt\\ml.txt'}, page_content='Machine Learning (ML) is a branch of Artificial Intelligence (AI) that focuses on enabling computers to learn from data and make decisions or predictions without being explicitly programmed for every task. Instead of following fixed instructions, ML systems identify patterns in data and improve their performance over time through experience.\n\nAt its core, machine learning uses algorithms to process large amounts of data, detect meaningful relationships, and generate models that can be used for tasks such as classification, prediction, and decision-making. These models are trained using historical data and can adapt as new data becomes available.\n\nThere are three main types of machine learning:\n\n1. Supervised Learning – In this approach, the model is trained on labeled data, where the correct output is already known. It is commonly used for tasks like spam detection and price prediction.\n\n2. Unsupervised Learning 

## pdf loader

In [83]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "data/pdf",
    glob="**/*.pdf", ##pattern to match files
    loader_cls=PyMuPDFLoader, ##loader class to use
)

directory_documnets = dir_loader.load()
directory_documnets

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': '', 'creationdate': 'D:20260428154453', 'source': 'data\\pdf\\Aryan_Patel_Professional_Portfolio.pdf', 'file_path': 'data\\pdf\\Aryan_Patel_Professional_Portfolio.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260428154453', 'page': 0}, page_content="Personal Portfolio and Career Vision - Aryan Patel\n 1. Who I Am: My Background and Philosophy\nMy name is Aryan Patel, and I am a Computer Engineering professional currently working at the intersection\nof Backend Development and Artificial Intelligence. Growing up with a natural curiosity for how things work, I\npursued a B.Tech in Computer Engineering from GEC Gandhinagar, where I maintained a strong academic\nrecord with a CGPA of 8.35. My journey in technology is driven by a simple belief: engineering is not just\nabout using t

In [84]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

def split_documnets(documents,chunk_size=1000,chunk_overlap=200):
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators= ["\n\n","\n"," ",""]
        )
    
    split_documnets = text_spliter.split_documents(documents)

    return split_documnets

In [85]:
chunks = split_documnets(directory_documnets)

In [86]:
chunks

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': '', 'creationdate': 'D:20260428154453', 'source': 'data\\pdf\\Aryan_Patel_Professional_Portfolio.pdf', 'file_path': 'data\\pdf\\Aryan_Patel_Professional_Portfolio.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260428154453', 'page': 0}, page_content='Personal Portfolio and Career Vision - Aryan Patel\n 1. Who I Am: My Background and Philosophy\nMy name is Aryan Patel, and I am a Computer Engineering professional currently working at the intersection\nof Backend Development and Artificial Intelligence. Growing up with a natural curiosity for how things work, I\npursued a B.Tech in Computer Engineering from GEC Gandhinagar, where I maintained a strong academic\nrecord with a CGPA of 8.35. My journey in technology is driven by a simple belief: engineering is not just\nabout using t

## EmBeddings And VectorStoreDB

In [87]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple, Any
from sklearn.metrics.pairwise import cosine_similarity

In [88]:
import os

In [89]:
class EmbeddingManager:
    def __init__(self, model_name:str = "all-MiniLM-l6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loadded successfully.")
            print(f"Embedding dimension: {self.model.get_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading Model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of text

        Args: 
            texts: List of text strings to embed

        returns: 
            numpy array of mebeddings with shape (len(texts), embedding_dim) 
        """
        if not self.model:
            raise ValueError('Model not loaded')
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self)-> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError('Model not loaded')
        return self.model.get_embedding_dimension()
    
## initialize the embedding manager

embeding_manager = EmbeddingManager()



loading embedding model : all-MiniLM-l6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1889.95it/s]


Model loadded successfully.
Embedding dimension: 384


## vector store

In [90]:
class VectoreStore:

    def __init__(self, collection_name: str = "txt_documents", persist_directory: str ="data/vector_store"):
        """ 
            Initialize the vectore store
            Args: 
                collection_name: name of the chromaDb collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize croma db client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"discription": "txt document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documnets in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise
    
    def add_documnets(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and its embeddings to store"""

        if len(embeddings) != len(documents):
            raise ValueError('Number of documents and embeddings do not match')

        print(f"Adding {len(documents)} documents to vector store")

        #prepare data for ChromaDb

        ids             = []
        metadatas       = []
        documents_text   = []
        embeddings_list = []

        for  i, (doc,embedding) in  enumerate(zip(documents,embeddings)):
            #generate unique Id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadta
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try: 
            self.collection.add(
                ids=ids,
                embeddings=embeddings,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documnets to vector store")
            print(f"total documents in collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documnets ito vector store: {e}")

vectorstore = VectoreStore()
vectorstore
    
        

Vector store initialized. Collection: txt_documents
Existing documnets in collection: 7


In [91]:
chunks

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': '', 'creationdate': 'D:20260428154453', 'source': 'data\\pdf\\Aryan_Patel_Professional_Portfolio.pdf', 'file_path': 'data\\pdf\\Aryan_Patel_Professional_Portfolio.pdf', 'total_pages': 2, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20260428154453', 'page': 0}, page_content='Personal Portfolio and Career Vision - Aryan Patel\n 1. Who I Am: My Background and Philosophy\nMy name is Aryan Patel, and I am a Computer Engineering professional currently working at the intersection\nof Backend Development and Artificial Intelligence. Growing up with a natural curiosity for how things work, I\npursued a B.Tech in Computer Engineering from GEC Gandhinagar, where I maintained a strong academic\nrecord with a CGPA of 8.35. My journey in technology is driven by a simple belief: engineering is not just\nabout using t

## convert text to embeddings

In [92]:
texts = [doc.page_content for doc in chunks]
texts

['Personal Portfolio and Career Vision - Aryan Patel\n 1. Who I Am: My Background and Philosophy\nMy name is Aryan Patel, and I am a Computer Engineering professional currently working at the intersection\nof Backend Development and Artificial Intelligence. Growing up with a natural curiosity for how things work, I\npursued a B.Tech in Computer Engineering from GEC Gandhinagar, where I maintained a strong academic\nrecord with a CGPA of 8.35. My journey in technology is driven by a simple belief: engineering is not just\nabout using tools that already exist, but about researching and creating new ones to solve real-world\nproblems.\nIn this document, I detail my current professional life, my technical background, my hands-on research\nprojects, and my long-term vision for contributing to the scientific landscape of India.\n 2. What I Am Doing: Professional Experience\nCurrently, I serve as a Backend Developer at Cybercom Creation, a role I began in January 2025. In this',
 '2. What I A

In [93]:
## generate the embeddings

embeddings = embeding_manager.generate_embeddings(texts)

## store in vector database
vectorstore.add_documnets(chunks, embeddings)

Generating embeddings for 9 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]

Generated embeddings with shape : (9, 384)
Adding 9 documents to vector store
Successfully added 9 documnets to vector store
total documents in collection : 16


## Retriver Pipeline From Vector Store

In [94]:
class RagRetriver:
    def __init__(self,vectore_store:VectoreStore,embeding_manager: EmbeddingManager):
        self.vector_store = vectore_store
        self.embedding_manager = embeding_manager

    def retrieve(self, query:str, top_k: int = 5, score_threshold : float = 0.0) -> List[Dict[str,Any]]:
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=top_k
            )

            retrived_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids       = results['ids'][0]

                for i,(doc_id, document, metadata, distance) in  enumerate(zip(ids, documents,metadatas,distances)):
                    #converts distance  to similarity scores
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance': distance,
                            'rank':i+1
                        })
                    print(f"Retrived {len(retrived_docs)} documents (after filtering)")
            else :
                print("No documnets found")

            return retrived_docs

        except Exception as e:
            print(f'error during retrival{e}')
            return []
        
rag_retriver = RagRetriver(vectorstore,embeding_manager)

In [57]:
rag_retriver

In [98]:
rag_retriver.retrieve("what is my Personal Interests and Soft Skills")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.73it/s]

Generated embeddings with shape : (1, 384)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)


[{'id': 'doc_c092c0db_4',
  'content': 'that autonomous systems and swarm robotics are the future of these sectors.\nSecond, I have a deep desire to give back to the academic community. After gaining sufficient experience in\nresearch and industry, I plan to focus on teaching. My goal is to pass down the knowledge and research\nhabits I acquire to enthusiastic students, helping them become the next generation of innovators. To achieve\nthese goals, I am pursuing advanced education (M.E./PhD) to move from being a developer to becoming a\nsubject-matter expert.\n 5. Personal Interests and Soft Skills\nI believe a balanced engineer is a creative engineer. Outside of technology, I am passionate about cooking,\nwhich allows me to experiment with different processes and results in a way that is similar to engineering. I\nhave also served as a campus ambassador for social causes, helping spread awareness about democratic\nparticipation. These activities have helped me develop strong communica

## integrate Vector DB context pipeline with llm output

In [111]:
import os 
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [110]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    max_tokens = 512
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [107]:
def rag_simple(query, retriver, llm,top_k = 3):
    ## retrive the context 
    results = retriver.retrieve(query,top_k)

    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "no relevent context found"

    ## generate output using llm

    prompt = f"""use this given context for the answer the question concisely.
        Context: 
        {context}

        question: 
        {query}"""
    
    response = llm.invoke([prompt.format(content=context,query=query)])

    return response.content

In [105]:
query = "what is my Personal Interests and Soft Skills"

## output with out any context

In [109]:
response = llm.invoke(query)
print(response.content)

As an AI, I don't have access to your personal information, so I can't tell you what your specific personal interests or soft skills are. I don't know anything about your experiences, preferences, or personality!

However, I can explain what these terms mean and provide common examples to help you reflect and identify your own.

---

### What are Personal Interests?

Personal interests are the activities, hobbies, topics, or subjects that you genuinely enjoy, are passionate about, and choose to spend your free time on. They are what energize you, make you curious, or bring you joy and satisfaction outside of your obligations.

**To identify your personal interests, ask yourself:**
*   What do you love doing when you have free time?
*   What topics do you enjoy learning about or discussing?
*   What activities make you lose track of time?
*   What have you always wanted to try or learn?
*   What problems do you enjoy solving, even for fun?

**Common Examples of Personal Interests:**

* 

## output with RAG context

In [108]:
answer = rag_simple(query,rag_retriver,llm)
answer

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.66it/s]

Generated embeddings with shape : (1, 384)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)
Retrived 1 documents (after filtering)


'Your personal interests include **cooking** and serving as a **campus ambassador for social causes**. Your soft skills are **communication** and **leadership**.'